# Human-in-the-Loop: Předakční brány, třídění rizika a auditní protokol

README k této lekci zavádí Human-in-the-Loop krátkým úryvkem, který po uživateli vyžaduje `APPROVE` nebo `REJECT` poté, co agent již vytvořil odpověď. Tento vzor je dobrý výchozí bod, ale produkční implementace HITL obvykle potřebují tři další součásti:

1. **Předakční brána**, která běží **před** tím, než agent provede riskantní krok, aby náklady, nezvratnost a latence zůstaly pod kontrolou.
2. **Třídění rizik**, takže nízkorizikové akce se automaticky spouštějí, středně rizikové akce se dávkově schvalují a pouze vysoce rizikové akce závisí na člověku.
3. **Auditní log plus revizní smyčka**, aby každé rozhodnutí brány bylo zaznamenáno jako JSONL a zamítnutí přesměrovávalo agenta s strukturovaným důvodem místo prostého zobrazení `Revising...`.

Tento notebook staví každou z těchto funkcí na stejných prvcích jako `06-system-message-framework.ipynb`. Funguje end-to-end v režimu `DEMO_MODE = True` (není potřeba interaktivní vstup) nebo s reálnými výzvami `input()`, když je `DEMO_MODE = False`. Poznámka: v DEMO_MODE je opakování třetího cíle scriptované, takže mechanika smyčky je viditelná end-to-end. Skutečná revizní přeuspořádání vyžadují `DEMO_MODE = False` a operátora.

**Mimo rozsah (řešeno v jiných lekcích):** autentizace a řízení přístupu (Lesson 06 README threat 2), middleware volání nástrojů (Lesson 14 MAF deep dive), vzory víceroagentových debat.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Vzor 1: Brána před akcí

Ukázka HITL v README nejdříve zavolá agenta, pak požádá uživatele o schválení výstupu. To je **post-akční** tok. Agent už provedl akci, takže náklady na volání LLM jsou již zaplaceny a jakýkoli vedlejší efekt (odeslaný e-mail, zapsaný řádek v databázi, zveřejněný komentář) už nastal.

**Pre-akční** tok vkládá bránu před spuštění riskantního kroku agentem. Agent navrhne akci, brána rozhodne, zda ji provést, a vedlejší efekt nastane pouze po schválení.

| Aspekt | Schválení po akci (ukázka README) | Brána před akcí (tento notebook) |
|---|---|---|
| Kdy probíhá schválení? | Po tom, co agent vytvoří výstup | Před vykonáním jakéhokoli vedlejšího efektu |
| Náklady na LLM při odmítnutí | Už zaplaceny | Zaplaceny jen za návrh, ne za vykonanou akci |
| Nevratné vedlejší efekty | Možné (akce už proběhla) | Zabráněno |
| Přehlednost auditu | Schválení je tisková zpráva | Schválení je záznam v JSON s časovou značkou, akcí a důvodem |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Vzor 2: Třídění podle rizika

Ne každá akce vyžaduje lidské schválení. Pouze čtecí dotaz na veřejné API má jiné riziko než odeslání e-mailu zákazníkovi. Zacházet s oběma stejně znamená plýtvat pozorností operátora a zpomaluje agenta.

Jednoduchý model se 3 úrovněmi:

| Úroveň | Příklady | Schvalovací postup |
|---|---|---|
| `nízká` (pouze čtení) | Vyhledávání v databázi znalostí, dohledání možností letu, načtení veřejné webové stránky | Automatické provedení, zaznamenáno pro audit |
| `střední` (levná mutace) | Uložení výsledku do mezipaměti, zvýšení čítače, naplánování připomenutí | Automatické provedení, ale hromadná denní kontrola |
| `vysoká` (externí nebo nezvratná) | Odeslání e-mailu, zpoplatnění karty, zveřejnění v kanálu | Zablokováno dokud nebude schváleno člověkem |

Tohle je jedno třídění. Produkční systémy často používají podrobnější úrovně (např. úrovně oprávnění AWS IAM, třídění přístupu založené na rolích). Níže uvedená verze se 3 úrovněmi je nejmenší užitečná verze pro agenta, který kombinuje pouze čtecí akce a akce s vedlejšími efekty.

Klasifikátor níže používá heuristiky na základě klíčových slov, aby demo zůstalo deterministické a levné. V produkčním systému byste jej nahradili naučeným klasifikátorem nebo politikovým enginem.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Vzor 3: Auditní záznam a smyčka revizí

`print("Response approved.")` není auditní záznam. Pro důvěryhodnost by každé rozhodnutí brány mělo být zaznamenáno jako strukturovaná událost, kterou lze později dotazovat, přehrát nebo připojit k přehledu incidentu.

Dvě části:

1. **Přidávající se JSONL.** Jeden řádek na rozhodnutí, s časovou značkou, akcí, úrovní, rozhodnutím, důvodem. Snadné hledání, snadné pozdější odeslání do skutečného logovacího systému.
2. **Smyčka revize při zamítnutí.** Když brána vrátí `deny`, agent se znovu vyzve s důvodem zamítnutí v kontextu, aby příští návrh mohl zabránit problému.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Další zdroje

Několik dalších veřejných projektů implementuje varianty těchto vzorů HITL. Porovnejte přístupy, abyste našli, co nejlépe vyhovuje vašemu zásobníku:

- **LangChain** nástroje s člověkem v cyklu ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): nástroje, které zastaví vykonávání pro lidský vstup.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ toto restrukturalizoval): používá roli agenta konkrétně k reprezentaci člověka v multiagentních konverzacích.
- **Semantic Kernel** filtry funkcí ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware, který běží kolem každého volání funkce, vhodný pro řízení logiky.

Každý projekt řeší tři podvzory jinak: LangChain je balí jako nástroje, AutoGen používá roli agenta, Semantic Kernel využívá filtry middleware. Přečtěte si jednu nebo dvě implementace kompletně, než si vyberete návrh pro svého vlastního agenta.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
